# EchoTwin — CPU demo on Colab

Runs the full **synthetic** pipeline end-to-end: WhatsApp parsing → dataset
→ style profile → `style_rag` mock baseline with contamination report →
voice recording plan + audio-quality analysis on a generated test tone.

No private data, no GPU, no model downloads. The repository is private, so
cell 1 asks for a GitHub personal access token (classic, `repo` scope) that
has access to it. The token is used only for `git clone` and never stored.

In [ ]:
import getpass
import os

token = getpass.getpass("GitHub token (repo scope, access to Shashwat1729/echotwin): ")
os.environ["GH_TOKEN"] = token
!git clone https://$GH_TOKEN@github.com/Shashwat1729/echotwin.git /content/echotwin
%cd /content/echotwin
!pip install -q -e .

## 1. Parse the synthetic export, pick the author, build the dataset

In [ ]:
import os

os.environ["ECHOTWIN_HOME"] = "/content/echotwin-home"
from echotwin.dataset import DatasetConfig, build_examples
from echotwin.dataset.splits import SplitConfig, split_dataset
from echotwin.ingestion import WhatsAppParser, participant_stats, pick_participant

with open("examples/synthetic_whatsapp.txt", encoding="utf-8") as fh:
    text = fh.read()
result = WhatsAppParser().parse_text(text, "demo")
print(f"parsed {len(result.messages)} messages, formats={result.report.detected_formats}")
for s in participant_stats(result.messages):
    print(f"- {s.sender_name}: {s.message_count} messages")
target = pick_participant(result.messages, "Aarav")
built = build_examples(result.messages, target, DatasetConfig())
split = split_dataset(built, SplitConfig())
print(f"examples={len(built.examples)} train/val/test="
      f"{len(split.train)}/{len(split.val)}/{len(split.test)}")
for w in split.report.warnings:
    print("WARNING:", w)

## 2. Style profile + `style_rag` mock baseline with contamination report

In [ ]:
from echotwin.dataset.retrieval import RetrievalConfig, build_index_from_examples
from echotwin.evaluation.baseline import heldout_prompts, run_baseline
from echotwin.models.generation import MockGenerator
from echotwin.profiling.style_profile import build_style_profile, profile_prompt_fragment

profile = build_style_profile(result.messages, target)
print("languages:", profile.language_use.get("distributions"))
prompts = heldout_prompts(split.test, max_prompts=20)
gen = MockGenerator([ex.messages[-1].content for ex in split.train[:5]], seed=7)
if prompts:
    records, path, report = run_baseline(
        method="style_rag",
        generator=gen,
        prompts=prompts,
        profile_fragment=profile_prompt_fragment(profile),
        output_dir="/content/echotwin-home/artifacts/evaluations/demo",
        model_revision="mock-1",
        retrieval_index=build_index_from_examples(built.examples),
        retrieval_cfg=RetrievalConfig(k=3),
        train_texts=[ex.messages[-1].content for ex in split.train],
    )
    print(f"predictions={len(records)} char3cos={report.style.char_ngram_cosine_3}")
    print("contamination:", report.contamination)
else:
    print("Single-conversation demo has no held-out split — import multiple chats for evaluation.")

## 3. Voice studio: plan, scripts, and quality analysis on a test tone

(Browser mic recording lives in the local Gradio UI; here we exercise the
same analysis code on an uploaded or generated clip.)

In [ ]:
import numpy as np

from echotwin.voice.audio_quality import analyze_wave_array, summary_line
from echotwin.voice.capabilities import VoiceBackendCapabilities
from echotwin.voice.recording_plan import RecordingPreferences, generate_plan
from echotwin.voice.registry import get_backend
from echotwin.voice.script_library import get_script, render

caps = VoiceBackendCapabilities(**get_backend("voxcpm").capabilities())
plan = generate_plan(caps, RecordingPreferences(languages=["hi", "en"]), [])
for t in plan.tasks[:6]:
    print(f"P{t.priority} [{'MANDATORY' if t.mandatory else 'optional'}] {t.title}: {t.script_ids}")
print()
print("Script hi-clean-01 reads:")
print(render(get_script("hi-clean-01"), {"NAME": "Demo"}))
print()
sr = 16000
t = np.arange(sr * 4) / sr
tone = (0.3 * np.sin(2 * np.pi * 180 * t) * np.clip(np.sin(2 * np.pi * 0.5 * t), 0, 1)).astype("float32")
rep = analyze_wave_array(tone, sr)
print(f"test tone: verdict={rep.verdict} peak={rep.peak} silence={rep.silence_fraction}")
print(summary_line(rep))

## Next steps

- Real training needs a GPU runtime (Colab GPU) + `pip install -e ".[train]"`
  and `python scripts/download_models.py --model qwen3-4b-it-2507`.
- Real voice synthesis needs `pip install voxcpm` + weights and a recorded
  voice profile with explicit consent — see `docs/voice-cloning.md`.